In [ ]:
import torch 
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(weights= models.ResNet50_Weights.DEFAULT).to(device)
model.eval()

incoming_requests = [torch.randn(3,224,224) for _ in range(8)]

batch_tensor = torch.stack(incoming_requests , dim=0).to(device)

with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=torch.float16):
  logits = model(batch_tensor)
  probabilites = torch.softmax(logits,dim=-1)
  predicted_classes = torch.argmax(probabilites,dim=-1)
  print(predicted_classes)


In [ ]:
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    # BasicBlock does not expand channels at the end of the block (64 -> 64)
    expansion = 1

    def __init__(self, in_channels: int, base_channels: int, stride: int = 1):
        super().__init__()
        out_channels = base_channels * self.expansion

        # Main Residual Branch F(x): Two 3x3 convolutions
        self.conv1 = nn.Conv2d(
            in_channels, base_channels, kernel_size=3, 
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(base_channels)
        
        self.conv2 = nn.Conv2d(
            base_channels, out_channels, kernel_size=3, 
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.relu = nn.ReLU(inplace=True)

        # Shortcut Branch: Identity by default, or 1x1 Projection if shapes change
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, kernel_size=1, 
                    stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        # 1. Compute F(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 2. Match identity shape if stride != 1 or channels changed
        if self.downsample is not None:
            identity = self.downsample(x)

        # 3. Residual connection + final activation
        out += identity
        out = self.relu(out)
        return out


class Bottleneck(nn.Module):
    # Bottleneck expands the internal base_channels by 4x at the final 1x1 conv
    expansion = 4

    def __init__(self, in_channels: int, base_channels: int, stride: int = 1):
        super().__init__()
        out_channels = base_channels * self.expansion

        # 1. Squeeze: 1x1 conv reduces channels (e.g., 256 -> 64)
        self.conv1 = nn.Conv2d(
            in_channels, base_channels, kernel_size=1, stride=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(base_channels)

        # 2. Spatial: 3x3 conv processes features (stride=2 happens here in ResNet v1.5!)
        self.conv2 = nn.Conv2d(
            base_channels, base_channels, kernel_size=3, 
            stride=stride, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(base_channels)

        # 3. Expand: 1x1 conv scales channels by 4x (e.g., 64 -> 256)
        self.conv3 = nn.Conv2d(
            base_channels, out_channels, kernel_size=1, stride=1, bias=False
        )
        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # Shortcut Branch: Triggered on spatial downsample OR channel expansion
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, kernel_size=1, 
                    stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))  # Note: No ReLU before adding identity!

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out

In [ ]:
def make_stage(block_class , in_channels, base_channels, num_blocks , stride:int = 1 ):
  layers = []
  
  # Block 0: Handles stride (downsampling) and channel expansion (uses 1x1 shortcut if needed)
  layers.append(block_class(in_channels , base_channels , stride=stride))

  # Update in_channels for the remaining blocks in this stage
  updated_in_channels = base_channels * block_class.expansion

  # Blocks 1 to N-1: Always stride=1 and pure identity skip connections (downsample=None)
  for _ in range(1 , num_blocks):
    layers.append(block_class(updated_in_channels , base_channels , stride = stride))

  
  return nn.Sequential(*layers)



In [ ]:
if __name__ == "__main__":
    x = torch.randn(2, 64, 56, 56)  # Batch=2, 64 channels from stem, 56x56 spatial

    # 1. ResNet-18 Stage 2 (2 BasicBlocks, base_channels=128, stride=2)
    stage2_r18 = make_stage(BasicBlock, in_channels=64, base_channels=128, num_blocks=2, stride=2)
    out_r18 = stage2_r18(x)
    print(f"ResNet-18 Stage 2 Output Shape : {out_r18.shape}")  # Expected: [2, 128, 28, 28]
    print(f"Block 0 downsample active?     : {stage2_r18[0].downsample is not None}") # True
    print(f"Block 1 downsample active?     : {stage2_r18[1].downsample is not None}") # False

    # 2. ResNet-50 Stage 2 (4 Bottlenecks, base_channels=128 -> 512 out, stride=2)
    # Assume input coming from ResNet-50 Stage 1 is [2, 256, 56, 56]
    x_r50 = torch.randn(2, 256, 56, 56)
    stage2_r50 = make_stage(Bottleneck, in_channels=256, base_channels=128, num_blocks=4, stride=2)
    out_r50 = stage2_r50(x_r50)
    print(f"\nResNet-50 Stage 2 Output Shape : {out_r50.shape}")  # Expected: [2, 512, 28, 28]

In [ ]:
# Fusing Conv2d + BatchNorm 
import torch
import torch.nn as nn

def manual_fuse_conv_bn(conv: nn.Conv2d, bn: nn.BatchNorm2d) -> nn.Conv2d:
    """
    Manually fuses an eval-mode Conv2d and BatchNorm2d into a single Conv2d with bias=True.
    """
    assert not (conv.training or bn.training), "Both layers must be in .eval() mode before fusion!"

    # 1. Create a new Conv2d layer with the exact same specs, EXCEPT bias=True
    fused_conv = nn.Conv2d(
        in_channels=conv.in_channels,
        out_channels=conv.out_channels,
        kernel_size=conv.kernel_size,
        stride=conv.stride,
        padding=conv.padding,
        dilation=conv.dilation,
        groups=conv.groups,
        bias=True,  # Crucial: BN's mean and beta require a bias term!
        padding_mode=conv.padding_mode
    ).to(conv.weight.device, conv.weight.dtype)

    # 2. Extract frozen BatchNorm parameters (all 1D tensors of shape [C_out])
    gamma = bn.weight
    beta = bn.bias
    mean = bn.running_mean
    var = bn.running_var
    eps = bn.eps

    # 3. Compute the per-channel scale factor: alpha = gamma / sqrt(var + eps)
    scale_factor = gamma / torch.sqrt(var + eps)  # Shape: [C_out]

    # 4. Fuse Weights: Conv weight has shape [C_out, C_in // groups, K_h, K_w]
    # Reshape scale_factor to [C_out, 1, 1, 1] so it broadcasts across each output filter
    with torch.no_grad():
        fused_conv.weight.copy_(conv.weight * scale_factor.view(-1, 1, 1, 1))

        # 5. Fuse Biases: If original conv had bias=False, treat b_conv as zeros
        b_conv = conv.bias if conv.bias is not None else torch.zeros_like(mean)
        fused_bias = (b_conv - mean) * scale_factor + beta
        fused_conv.bias.copy_(fused_bias)

    return fused_conv


# --- Verification & Comparison with PyTorch Built-in ---
if __name__ == "__main__":
    torch.manual_seed(42)

    # Setup a Conv + BN pair (like ResNet's first layer)
    conv = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False)
    bn = nn.BatchNorm2d(128)

    # Simulate a few training forward passes so running_mean and running_var aren't just 0 and 1
    for _ in range(5):
        _ = bn(conv(torch.randn(4, 64, 28, 28)))

    # Switch to evaluation mode (MANDATORY before fusing)
    conv.eval()
    bn.eval()

    # 1. Fuse using our manual math
    my_fused_conv = manual_fuse_conv_bn(conv, bn)

    # 2. Fuse using PyTorch's built-in utility
    pytorch_fused_conv = nn.utils.fuse_conv_bn_eval(conv, bn)

    # 3. Verify outputs match to floating-point precision (~1e-6)
    test_input = torch.randn(2, 64, 28, 28)
    with torch.inference_mode():
        out_unfused = bn(conv(test_input))
        out_manual  = my_fused_conv(test_input)
        out_pytorch = pytorch_fused_conv(test_input)

    print(f"Max diff (Unfused vs Manual)  : {(out_unfused - out_manual).abs().max().item():.2e}")
    print(f"Max diff (Manual  vs PyTorch) : {(out_manual - out_pytorch).abs().max().item():.2e}")
    

In [ ]:
def fuse_resnet_block(block: nn.Module):
    """Fuses all Conv+BN pairs inside a BasicBlock or Bottleneck in-place."""
    block.eval()
    
    # Fuse main branch convolutions
    block.conv1 = nn.utils.fuse_conv_bn_eval(block.conv1, block.bn1)
    block.bn1 = nn.Identity()

    block.conv2 = nn.utils.fuse_conv_bn_eval(block.conv2, block.bn2)
    block.bn2 = nn.Identity()

    if hasattr(block, "conv3"):  # If it's a Bottleneck block
        block.conv3 = nn.utils.fuse_conv_bn_eval(block.conv3, block.bn3)
        block.bn3 = nn.Identity()

    # Don't forget the 1x1 projection shortcut if present!
    if block.downsample is not None:
        block.downsample[0] = nn.utils.fuse_conv_bn_eval(
            block.downsample[0], block.downsample[1]
        )
        block.downsample[1] = nn.Identity()

In [ ]:
def fuse_resnet_block(block: nn.Module):
    """Fuses all Conv+BN pairs inside a BasicBlock or Bottleneck in-place."""
    block.eval()
    
    # Fuse main branch convolutions
    block.conv1 = nn.utils.fuse_conv_bn_eval(block.conv1, block.bn1)
    block.bn1 = nn.Identity()

    block.conv2 = nn.utils.fuse_conv_bn_eval(block.conv2, block.bn2)
    block.bn2 = nn.Identity()

    if hasattr(block, "conv3"):  # If it's a Bottleneck block
        block.conv3 = nn.utils.fuse_conv_bn_eval(block.conv3, block.bn3)
        block.bn3 = nn.Identity()

    # Don't forget the 1x1 projection shortcut if present!
    if block.downsample is not None:
        block.downsample[0] = nn.utils.fuse_conv_bn_eval(
            block.downsample[0], block.downsample[1]
        )
        block.downsample[1] = nn.Identity()